# 04 · Risk Metrics & Attribution

Valuation results are only the starting point. This notebook demonstrates how to harvest standardized metrics (DV01, duration, spreads), compute laddered risk (KRD/CS01), and explain P&L between two dates via the attribution engine.

### Learning Objectives
- Explore `MetricRegistry` to discover which analytics apply to a given instrument
- Run pricing with a curated metric set to obtain DV01, yield, spread, and convexity measures
- Generate bucketed interest-rate and credit ladders with `finstack.valuations.risk`
- Attribute P&L between T₀/T₁ using parallel and waterfall methodologies

In [ ]:
from datetime import date, timedelta
import math

from finstack import Money
from finstack.core.currency import USD
from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve
from finstack.valuations.instruments import Bond
from finstack.valuations.metrics import MetricRegistry, MetricId
from finstack.valuations.pricer import create_standard_registry
from finstack.valuations.attribution import AttributionMethod, attribute_pnl

as_of = date(2025, 1, 2)

def build_discount_curve(as_of: date, shift_bp: float = 0.0) -> DiscountCurve:
    base = [
        (0.0, 1.0),
        (0.5, 0.9975),
        (1.0, 0.9940),
        (2.0, 0.9850),
        (5.0, 0.9500),
        (10.0, 0.8900),
    ]
    if abs(shift_bp) < 1e-9:
        points = base
    else:
        shift = shift_bp / 10_000.0
        points = []
        for t, df in base:
            if t == 0.0:
                points.append((t, df))
                continue
            zero = -math.log(df) / t
            new_zero = zero + shift
            points.append((t, math.exp(-new_zero * t)))
    return DiscountCurve("USD-OIS", as_of, points)

# Shared bond instrument
notional = Money(25_000_000, USD)
bond = Bond.fixed_semiannual(
    "CORP-5Y",
    notional,
    0.045,
    issue=date(2024, 1, 2),
    maturity=date(2029, 1, 2),
    discount_curve="USD-OIS",
)

# Baseline market context
market = MarketContext()
market.insert_discount(build_discount_curve(as_of))
registry = create_standard_registry()


## 1. Discover Metrics With `MetricRegistry`
`MetricRegistry.standard()` exposes every metric the engine can compute. Filter by instrument type to understand what's available before requesting them in `price_with_metrics`.

In [ ]:
metric_registry = MetricRegistry.standard()
available = metric_registry.metrics_for_instrument("bond")
print("Total metrics registered:", len(metric_registry.available_metrics()))
print("Bond-specific metrics (subset):", [m.name for m in available[:8]])
print("Has z-spread?", metric_registry.has_metric("z_spread"))
print("Yield metric applicable?", metric_registry.is_applicable("ytm", "bond"))


## 2. Price With Metrics
Request a handful of analytics during pricing. Metrics are computed in the engine (no reprojection required) and returned in `result.measures`.

In [ ]:
metric_names = [
    "clean_price",
    "dirty_price",
    "accrued",
    "ytm",
    "duration_mod",
    "dv01",
    "z_spread",
    "i_spread",
    "convexity",
]
result = registry.price_with_metrics(
    bond,
    "discounting",
    market,
    metric_names,
    as_of=as_of,
)
print("Present value:", result.value)
for name in metric_names:
    print(f"{name:>14}:", result.measures.get(name))


## 3. Bucketed Interest-Rate and Credit Risk
`finstack.valuations.risk` offers canned ladders for DV01 (key-rate duration) and CS01. Results are dictionaries that can be promoted into pandas/Polars data frames for visualization.

In [ ]:
krd_result = registry.price_with_metrics(
    bond,
    "discounting",
    market,
    ["bucketed_dv01"],
    as_of=as_of,
)

krd_result.to_dict()


## 4. P&L Attribution Between Two Dates
Attribution reprices at T₀ and T₁, isolating factor moves. We create shifted markets to emulate a +5 bp rates move between 2025-01-02 and 2025-02-02 and compare the parallel and waterfall methodologies.

In [ ]:
market_t0 = MarketContext()
market_t0.insert_discount(build_discount_curve(as_of, shift_bp=0.0))

market_t1 = MarketContext()
market_t1.insert_discount(build_discount_curve(as_of, shift_bp=5.0))

as_of_t0 = as_of
as_of_t1 = as_of + timedelta(days=31)

attr_parallel = attribute_pnl(
    bond,
    market_t0,
    market_t1,
    as_of_t0,
    as_of_t1,
    method=AttributionMethod.parallel(),
)
print("Parallel attribution")
print("  Total P&L:", attr_parallel.total_pnl)
print("  Carry:", attr_parallel.carry)
print("  Rates contribution:", attr_parallel.rates_curves_pnl)
print("  Residual (% of total):", attr_parallel.residual, f"({attr_parallel.meta.residual_pct:.2%})")

attr_waterfall = attribute_pnl(
    bond,
    market_t0,
    market_t1,
    as_of_t0,
    as_of_t1,
    method=AttributionMethod.waterfall(["carry", "rates_curves"]),
)
print("Waterfall attribution (carry → rates)")
print("  Total P&L:", attr_waterfall.total_pnl)
print("  Carry first:", attr_waterfall.carry)
print("  Rates second:", attr_waterfall.rates_curves_pnl)


## Summary
- `MetricRegistry` surfaces supported analytics upfront, avoiding guesswork.
- `price_with_metrics` produces PV plus sensitivity measures in one call.
- Ladder utilities turn risk bumps into structured datasets for reporting.
- Attribution decomposes realized P&L using either independent (parallel) or ordered (waterfall) factor treatments, helping explain daily moves to stakeholders.